# Chroma Vector Database
- Chroma는 대규모 언어 모델(LLM) 애플리케이션 구축을 위해 설계된 AI 네이티브 **오픈 소스 벡터 데이터베이스**다.    
- 임베딩 저장소, 쿼리 및 검색 등의 핵심 기능을 제공하여 개발자들이 효율적으로 작업할 수 있도록 돕는다. 
- https://www.trychroma.com/
  
## Chroma의 주요 특징

- **오픈 소스 라이선스** 
  - Apache 2.0 라이선스에 따라 제공되어 누구나 자유롭게 사용하고 수정할 수 있다. 
- **다양한 개발 환경 지원**
  -  Python 및 JavaScript/TypeScript SDK를 지원하여 다양한 Langchain 과 연동하여 활용할 수 있다. 
- **유연한 데이터 저장 옵션**
  -  HTTP 방식, 디스크 저장 방식, 인메모리 방식을 선택하여 데이터를 저장할 수 있어 사용자 입장에서 매우 편리하다. 
- **간편한 사용법** 
  - 설치 및 사용법이 매우 간단하여 빠르게 프로토타입을 개발하고 검증할 수 있다. 

## 설치
- pip로 chromadb 설치시 **windows**에서는 c컴파일러 관련되어 에러가 난다. **conda 를 이용해 설치한다.**
- `conda install conda-forge::chromadb`
- `pip install langchain-chroma`

# Chroma API 를 이용해 연동
- https://docs.trychroma.com/

In [1]:
import chromadb

In [1]:
from uuid import uuid4
uuid4()

UUID('f381904f-88e2-4157-ba8d-9855dc61699b')

In [2]:
from uuid import uuid4

# 추가할 데이터
document_list = [
        "This is a document about pineapple",
        "This is a document about oranges",
        "This is a document about sports",
        "This is a document about langchain",
]
ids = [str(uuid4()) for _ in range(len(document_list))]
# DB에 저장할 때 지정할 각 문서들의 ID 생성

In [3]:
# 외부 embeding model
from dotenv import load_dotenv
import chromadb.utils.embedding_functions as embedding_functions
import os

print(load_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)

True


In [4]:
from langchain_openai import OpenAIEmbeddings
# embedding model 생성
embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small")

In [ ]:
# Collection : DB
# Chroma DB 연결 
import chromadb
client = chromadb.Client()	# InMemory DB (data를 memory에 저장.)
# client = chromadb.PersistentClient(path="vector_store/chroma/my_db")	# Local file에 저장.
# client = chromadb.HttpClient(host="ip 주소", port=포트번호)		# server로 서비스하는 chromadb에 연결.

# Collection 생성
collection_name = "test_db"
collection = client.create_collection(
    name = collection_name,
    get_or_create = True,	# collection이 있으면 연결, 없으면 생성 (default : False -> 이미 있는 collection이면 Exception 발생.)
    metadata = {"hnsw:space":"cosine"},	# cosine 유사도로 계산.
    embedding_function=openai_ef
)

In [8]:
# data 추가
collection.add(documents=document_list, ids=ids)

In [9]:
# 유사도 검색
result = collection.query(
    query_texts = ["deeplarning"],	# 질문
    n_results = 2				# 검색 결과 수
)
result

{'ids': [['458ddd6d-d6ea-4eb8-abbd-c533ea9a7813',
   '422f8359-7f62-4122-946a-45a36281695e']],
 'embeddings': None,
 'documents': [['This is a document about langchain',
   'This is a document about pineapple']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.7646393775939941, 0.8501489162445068]]}

# Langchain을 이용해 Chroma 연동

## Data 준비

In [20]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
    id=1,
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
    id=2,
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
    id=3,
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
    id=4,
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
    id=5,
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
    id=6,
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
    id=7,
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
    id=8,
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
    id=9,
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
    id=10,
)
document_list = [document_1, document_2, document_3, document_4, document_5, document_6, document_7, document_8, document_9, document_10]
ids = [str(uuid4()) for _ in range(len(document_list))]

In [15]:
!pip install langchain-chroma

## Vector Store 생성, 연결
- Chroma.from_documents()
  - VectorStore를 초기화(생성)하고 문서를 추가한다.
  - persist_directory를 지정하지 않으면 메모리에 저장된다.

In [22]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

COLLECTION_NAME = "example"		# collenction (RDB의 DB) 이름
PERSISTENT_PATH = "vector_store/chroma/example_db"	# 저장할 loacl path

embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small")

# 1. 연결(생성)하면서 document들을 저장(upsert)
vector_store = Chroma.from_documents(
    documents=document_list,
    ids=ids,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSISTENT_PATH
)


In [24]:
# 2. 연결
vector_store2 = Chroma(
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSISTENT_PATH
)


TypeError: Chroma.__init__() got an unexpected keyword argument 'embedding'

## VectorStore 정보 확인

In [26]:
coll = vector_store._collection
coll, coll.count()	# 저장된 data수

(Collection(name=example), 10)

## Add (추가)

In [27]:
document_11 = Document(
    page_content="랭체인은 대규모 언어 모델(LLM)을 효과적으로 활용하기 위한 도구와 프레임워크를 제공하는 오픈소스 라이브러리입니다.",
    metadata={"source": "tweet"},
    id=10,
)

document_12 = Document(
    page_content="랭체인은 체인 구조를 사용하여 여러 LLM 작업을 연결하고, 이를 통해 더 복잡하고 맞춤화된 자연어 처리 애플리케이션을 개발할 수 있게 합니다",
    metadata={"source": "tweet"},
    id=10,
)

document_13 = Document(
    page_content="랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게!",
    metadata={"source": "news"},
    id=10,
)

In [28]:
vector_store.add_documents([document_11, document_12, document_13], ids = [str(uuid4()), str(uuid4()), str(uuid4())])

['ba6dd7bc-1c06-4b16-bfce-7b33cb64b7a5',
 '4c0cbb69-87e9-467e-9227-b65003361aa0',
 '14d8aa4a-fe58-48ad-b9f5-ad3df450a3ef']

In [29]:
coll.count()

13

## Update(갱신)

In [30]:
new_document_13 = Document(
    page_content="랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게! 아우 힘들어 피곤해 어려워",
    metadata={"source": "news"},
    id=10,
)

In [36]:
vector_store.update_document(
    document_id="14d8aa4a-fe58-48ad-b9f5-ad3df450a3ef",	# 바꿀 문서의 ID
    document=new_document_13	# 바꿀 내용을 가진 document 객체
)

In [38]:
update_document_12 = Document(
    page_content="랭체인은 체인 구조를 사용하여 여러 LLM 작업을 연결하고, 이를 통해 더 복잡하고 맞춤화된 자연어 처리 애플리케이션을 개발할 수 있게 합니다",
    metadata={"source": "website"}
)

update_document_13 = Document(
    page_content="랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게!",
    metadata={"source": "news"}
)
update_docs = [update_document_12, update_document_13]
update_ids = ['4c0cbb69-87e9-467e-9227-b65003361aa0', '14d8aa4a-fe58-48ad-b9f5-ad3df450a3ef']
# 한꺼번에 update
vector_store.update_documents(documents=update_docs, ids=update_ids)

In [39]:
coll.get()	# 전체 저장된 문서 조회 (select * from table)
vector_store.get()

{'ids': ['4d061638-4f3c-47a0-98d6-2a8700892fbc',
  'f230c848-f99b-4d0d-8eb4-cd2c9ab38209',
  '9ced2452-bd26-4a8f-b30e-ed9179f8460e',
  '6048c38d-8300-44b1-a254-881561ad393c',
  '9cf337ef-1368-44e0-ae71-cd7e1cf496b2',
  'a10ae603-154e-4a64-a1f7-c641e33e480a',
  '409b3a69-cc97-4deb-b97c-4b48ceee50af',
  'd88979bd-e0f0-4c13-a7db-8b57e59692fa',
  'e09343e4-9a7a-4e93-b0f6-2b86aab3558c',
  'de1a22d8-e1f7-4fa7-af28-2d27a6457813',
  'ba6dd7bc-1c06-4b16-bfce-7b33cb64b7a5',
  '4c0cbb69-87e9-467e-9227-b65003361aa0',
  '14d8aa4a-fe58-48ad-b9f5-ad3df450a3ef'],
 'embeddings': None,
 'documents': ['I had chocolate chip pancakes and scrambled eggs for breakfast this morning.',
  'The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.',
  'Building an exciting new project with LangChain - come check it out!',
  'Robbers broke into the city bank and stole $1 million in cash.',
  "Wow! That was an amazing movie. I can't wait to see it again.",
  'Is the new iPhone worth the 

## Delete(삭제)

In [42]:
del_ids =['a10ae603-154e-4a64-a1f7-c641e33e480a','409b3a69-cc97-4deb-b97c-4b48ceee50af']	# [삭제할 문서들의 id들]
vector_store.delete(ids = del_ids)

In [41]:
coll.count()

11

## Query(조회)
- `similarity_search(query, k, filter)`
  - 저장되 있는 item들 중 질의와 가장 유사한 것 k개를 찾는다. 
  - 찾은 결과를 filter 조건으로 필터링 한다. filter 조건은 meta-data의 정보를 이용한다.
  - 질의어(query)는 text(자연어)로 입력한다.
- `similarity_search_with_score(query, k, filter)`
  - 저장되 있는 item들 중 질의와 가장 유사한 것 k개를 찾아 유사도 점수와 함께 반환
- `similarity_search_by_vector(embedding, k, filter)`
  - Embedding Vector 를 질의로 입력한다. (질의(query)를 문장이 아니라 embedding vector로 입력.) 

In [43]:
results = vector_store.similarity_search(
    query = "Langchain이란 무엇인가욥 ?",
    k = 3,	# 조회 개수
)
results

[Document(id='9ced2452-bd26-4a8f-b30e-ed9179f8460e', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='ba6dd7bc-1c06-4b16-bfce-7b33cb64b7a5', metadata={'source': 'tweet'}, page_content='랭체인은 대규모 언어 모델(LLM)을 효과적으로 활용하기 위한 도구와 프레임워크를 제공하는 오픈소스 라이브러리입니다.'),
 Document(id='d88979bd-e0f0-4c13-a7db-8b57e59692fa', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [ ]:
results = vector_store.similarity_search_with_score(
    query = "아침에 무엇을 먹는게 좋을까요 ?",
    k = 3,	# 조회 개수
)

results

[(Document(id='4d061638-4f3c-47a0-98d6-2a8700892fbc', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
  1.2218379974365234),
 (Document(id='14d8aa4a-fe58-48ad-b9f5-ad3df450a3ef', metadata={'source': 'news'}, page_content='랭체인, AI 활용의 새 시대를 열다: 복잡한 언어 처리도 간단하게!'),
  1.7356758117675781),
 (Document(id='f230c848-f99b-4d0d-8eb4-cd2c9ab38209', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
  1.745650291442871)]

In [46]:
results = vector_store.similarity_search_with_score(
    query = "아침에 무엇을 먹는게 좋을까요 ?",
    k = 3,	# 조회 개수
    filter = {"source" : "tweet"}	# matadata의 source키 값이 tweet(source == tweet)
)
# 1. filter에 설정과 metadata를 비교해서 조회
# 2. 1에서 조회된 문서들과 query간의 유사도를 체크
results

[(Document(id='4d061638-4f3c-47a0-98d6-2a8700892fbc', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
  1.2218379974365234),
 (Document(id='ba6dd7bc-1c06-4b16-bfce-7b33cb64b7a5', metadata={'source': 'tweet'}, page_content='랭체인은 대규모 언어 모델(LLM)을 효과적으로 활용하기 위한 도구와 프레임워크를 제공하는 오픈소스 라이브러리입니다.'),
  1.8468457460403442),
 (Document(id='d88979bd-e0f0-4c13-a7db-8b57e59692fa', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
  1.8888370990753174)]

In [ ]:
results = vector_store.similarity_search_with_score(
    query = "아침에 무엇을 먹는게 좋을까요 ?",
    k = 3,	# 조회 개수
    filter = {"source" : {"$ne":"news"}}	# matadata의 source키 값이 news가 아닌 것들
    # {metadata key : {"연산자" : "값"}}
    # {"age : {"gt", 30}"}	# age > 30
)
# 1. filter에 설정과 metadata를 비교해서 조회
# 2. 1에서 조회된 문서들과 query간의 유사도를 체크
results

[(Document(id='4d061638-4f3c-47a0-98d6-2a8700892fbc', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
  1.2218379974365234),
 (Document(id='ba6dd7bc-1c06-4b16-bfce-7b33cb64b7a5', metadata={'source': 'tweet'}, page_content='랭체인은 대규모 언어 모델(LLM)을 효과적으로 활용하기 위한 도구와 프레임워크를 제공하는 오픈소스 라이브러리입니다.'),
  1.8468457460403442),
 (Document(id='4c0cbb69-87e9-467e-9227-b65003361aa0', metadata={'source': 'website'}, page_content='랭체인은 체인 구조를 사용하여 여러 LLM 작업을 연결하고, 이를 통해 더 복잡하고 맞춤화된 자연어 처리 애플리케이션을 개발할 수 있게 합니다'),
  1.8709466457366943)]